# Machine Learning #03 — Convolutional Neural Networks in PyTorch 🖼️

In the previous lessons we built **fully connected** networks and wrote the
**training loop** ourselves. Today we reuse tensors, `nn.Module`, and the
loss/optimizer/loop, and introduce **convolutional** layers (`nn.Conv2d`) and
**pooling** (`nn.MaxPool2d`) to extract features from images. We still use
`nn.Linear` layers at the end to classify those features.

> **The plan for today:**
> **understand convolution → build a CNN → train it → evaluate → look inside it 🔬**

## Why not just a plain neural network? 🤔

A 28×28 grayscale image has **784 pixel values**. A colour 32×32 image has
**1,024 pixels**, each with three RGB values: **3,072 input features**.
A fully connected network can classify images, but it has two limitations:

- **Many weights.** A `Linear(784, 256)` layer has 200,704 weights plus 256 biases.
  The parameter count depends on both the input size and the number of neurons.
- **No built-in spatial structure.** Flattening preserves every pixel value, but
  a linear layer does not explicitly use the image's 2-D neighbourhoods or share
  a detector across locations.

A **Convolutional Neural Network (CNN)** slides small learned filters over the
image and reuses the same weights at each location. This makes local patterns
such as edges useful across different parts of the image.

| | Fully connected (Lesson 2) | Convolutional (today) |
|---|---|---|
| Input | a flat vector of features | an image with spatial dimensions and channels |
| Core layer | `nn.Linear` | **`nn.Conv2d`** |
| Weights | one weight per input-output connection | shared local filters |
| Uses 2-D neighbourhoods explicitly? | no | yes |

## Imports and setup ⚙️

For faster training in Colab, select *Runtime → Change runtime type → GPU*.
The notebook also works on CPU. Locally, install the project dependencies with
`python -m pip install .` from the repository root. The next cell installs the
extra packages used here into the active notebook kernel, including in Colab.

In [ ]:
%pip install -q torchinfo scikit-image scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split

import torchvision
from torchvision import datasets, transforms

import torchinfo

from sklearn.metrics import accuracy_score, confusion_matrix

sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version:", torch.__version__)
print("device:", device)

# 📌 Part 1 — Understanding convolution

## 📒 Convolution

A **convolution** slides a small matrix — a **kernel** (also called a *filter* or
*mask*) — over the image. At every position it multiplies the kernel with the
pixels underneath and sums the result into a single output pixel.

In image processing it is defined as:

$$g(x, y) = \omega * f(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b}\omega(s,t)\,f(x-s, y-t)$$

where $f(x,y)$ is the original image, $\omega$ is the kernel, and $g(x,y)$ is the
filtered image.

**Technical detail:** mathematical convolution flips the kernel before
sliding it across the image. PyTorch
[`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
uses **cross-correlation**, which does not flip it. The blur, edge-detection,
and sharpen kernels used below are symmetric under a 180° rotation, so both
operations give the same result for them. With learned filters, either
convention can represent the same operations.

![Example of the Convolution](https://github.com/rasvob/VSB-FEI-Deep-Learning-Exercises/blob/main/images/convolution_example.gif?raw=true "Conv eg")

### 💡 Example filters

Different kernels extract different things. Notice these are **hand-designed** —
later the network will **learn** its own filters automatically. 🎯

|Name|Definition|
|----|:--------:|
|Identity| $\left[\begin{matrix}0&0&0\\0&1&0\\0&0&0\end{matrix}\right]$|
|Sobel vertical edges| $\left[\begin{matrix}+1&0&-1\\+2&0&-2\\+1&0&-1\end{matrix}\right]$|
|Sobel horizontal edges| $\left[\begin{matrix}+1&+2&+1\\0&0&0\\-1&-2&-1\end{matrix}\right]$|
|Edge detection| $\left[\begin{matrix}-1&-1&-1\\-1&8&-1\\-1&-1&-1\end{matrix}\right]$|
|Sharpen| $\left[\begin{matrix}0&-1&0\\-1&5&-1\\0&-1&0\end{matrix}\right]$|
|Uniform blur|$\frac{1}{9}\left[\begin{matrix}1&1&1\\1&1&1\\1&1&1\end{matrix}\right]$|
|Gaussian blur 3×3| $\frac{1}{16}\left[\begin{matrix}1&2&1\\2&4&2\\1&2&1\end{matrix}\right]$|

## 🚀 Let's apply filters ourselves — no network yet

Before we let a network learn filters, let's **be the filter** and apply a few by
hand. We use a classic grayscale test image (*"cameraman"*, bundled with
`scikit-image`) and the plain `scipy.signal.convolve2d` function.

In [ ]:
from skimage import data                 # bundled sample images, no download
from scipy.signal import convolve2d

img = data.camera()                        # 512x512 grayscale "cameraman" image
img = img / 255.0                          # normalise pixels to the 0-1 range
print("image shape:", img.shape)

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray")
plt.title("Original")
plt.axis("off")
plt.show()

In [ ]:
# Define a few 3x3 kernels by hand
blur_mask = np.ones((3, 3)) / 9.0          # uniform blur (average of neighbours)

edge_mask = np.ones((3, 3)) * -1
edge_mask[1, 1] = 8                        # edge detection (centre 8, neighbours -1)

sharpen_mask = np.array([[ 0, -1,  0],
                         [-1,  5, -1],
                         [ 0, -1,  0]])    # sharpen

print("edge kernel:\n", edge_mask)

In [ ]:
# convolve2d slides the kernel across the whole image.
#   boundary='symm' -> mirror the image at the borders (so the filter has values to read)
#   mode='same'     -> output keeps the same size as the input
img_blur    = convolve2d(img, blur_mask,    boundary="symm", mode="same")
img_edges   = convolve2d(img, edge_mask,    boundary="symm", mode="same")
img_sharp   = convolve2d(img, sharpen_mask, boundary="symm", mode="same")

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, title in zip(
        axes,
        [img, img_blur, np.clip(img_edges, 0, 1), np.clip(img_sharp, 0, 1)],
        ["Original", "Blur", "Edge detection", "Sharpen"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

> 💡 **Key idea:** each kernel highlights something different (smoothing, edges,
> detail). A CNN does exactly this — but it **learns the numbers inside the kernels
> from data** instead of us choosing them.

## 📒 Padding

When a 3×3 filter slides over the image it **cannot start right at the edge** —
there are no neighbours outside the border. Without help, the output shrinks a
little with every convolution.

**Padding** adds a border of extra (usually zero) pixels around the image so the
output can keep the same size.

![Padding example](https://github.com/rasvob/VSB-FEI-Deep-Learning-Exercises/blob/main/images/padding_example.png?raw=true)

For a **3×3 kernel with stride 1 and dilation 1**, set padding on the layer:

- `nn.Conv2d(..., padding=0)` → **valid** convolution, the image shrinks.
- `nn.Conv2d(..., padding=1)` → adds a 1-pixel border, a 3×3 filter **keeps the
  size** (equivalent to `padding='same'` for these settings).

Larger kernels, strides, or dilation values change the output-size calculation.

## 📒 Pooling

**Pooling** reduces spatial size by summarising local windows. Max pooling
keeps the strongest activation in each window; average pooling keeps the mean.

- **Max pooling** takes the maximum of each small window (usually 2×2). This is the
  standard choice in hidden layers.
- **Average pooling** takes the mean; it is rarer, mostly used near the very end.

![MaxPooling example](https://github.com/rasvob/VSB-FEI-Deep-Learning-Exercises/blob/main/images/pooling_example.png?raw=true)

Pooling can give **some tolerance to small shifts**, for example when a maximum
stays inside the same pooling window. It does **not** guarantee translation
invariance: moving a feature across window boundaries can change the output
substantially. See this [study of shift sensitivity in CNNs](https://proceedings.mlr.press/v97/zhang19a.html).

## ⚡ Understanding layer dimensions

Keep track of channels and spatial dimensions separately:

**Convolution (`nn.Conv2d`) 🔍**
- **Channels** become `out_channels`, the number of filters. They can increase
  (`1 → 32`), decrease (`32 → 16`), or stay the same.
- For a **3×3 kernel, stride 1, and dilation 1**, width and height stay the same
  with `padding=1`, or shrink by 2 with `padding=0`.

**Pooling (`nn.MaxPool2d(2)`) 🔄**
- **Width and height** are halved, rounding down for odd sizes (`11 → 5`).
- **Channels** stay unchanged.

> 💡 **Conv → learned features**, **Pool → smaller spatial dimensions**.
> Our example repeats these operations, then flattens the result and classifies it.

# 📦 Part 2 — The data

We use **FashionMNIST**: 70,000 grayscale **28×28** images of clothing in **10
classes** (t-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag,
ankle boot).


## Load the data

In [ ]:
# transforms.ToTensor() converts a PIL image to a float tensor of shape (1, 28, 28)
# with pixel values scaled to the 0-1 range.
transform = transforms.ToTensor()

full_train = datasets.FashionMNIST(root="data", train=True,  download=True, transform=transform)
test_set   = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

# Split the training data into a real training set and a validation set (80/20).
n_val = int(0.2 * len(full_train))
n_train = len(full_train) - n_val
train_set, val_set = random_split(full_train, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(SEED))

class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("\n train:", len(train_set), " val:", len(val_set), " test:", len(test_set))

In [ ]:
# A DataLoader serves the data in mini-batches (small chunks) and shuffles it.
# This is the PyTorch way of feeding data to the training loop.
BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
# Clean training examples for loss comparisons after each epoch.
train_eval_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

# Peek at one batch to check the shapes.
images, labels = next(iter(train_loader))
print("batch of images:", images.shape)   # (64, 1, 28, 28) = (batch, channels, H, W)
print("batch of labels:", labels.shape)   # (64,)

## A quick look at the data 🔎

In [ ]:
plt.figure(figsize=(9, 9))
for i in range(min(25, len(images))):
    plt.subplot(5, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(images[i][0], cmap="gray")     # [0] = the single grayscale channel
    plt.xlabel(class_names[labels[i]])
plt.tight_layout()
plt.show()

# 🧠 Part 3 — Build the CNN

We reuse the **`nn.Module` class** style from Lesson 1. The only new pieces are
`nn.Conv2d` and `nn.MaxPool2d`. Our tiny network is:

$$\text{image (1×28×28)} \rightarrow \text{Conv+ReLU+Pool} \rightarrow \text{Conv+ReLU+Pool} \rightarrow \text{Flatten} \rightarrow \text{Linear+ReLU} \rightarrow \text{Linear} \rightarrow \text{10 logits}$$


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()                          # required first line
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)   # 1 -> 32 channels
        self.conv2 = nn.Conv2d(32, 16, kernel_size=3)  # 32 -> 16 channels
        self.pool  = nn.MaxPool2d(2)                    # halves width & height
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * 5 * 5, 64)            # 400 -> 64
        self.fc2 = nn.Linear(64, n_classes)             # 64 -> 10 logits

    def forward(self, x):                            # describe the data flow
        x = self.pool(torch.relu(self.conv1(x)))     # conv -> relu -> pool
        x = self.pool(torch.relu(self.conv2(x)))     # conv -> relu -> pool
        x = self.flatten(x)                          # 2-D maps -> 1-D vector
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)                              # raw logits (no softmax here)
        return x

model = SimpleCNN().to(device)
print(model)

In [ ]:
# torchinfo prints the shape after every layer and the parameter count.
torchinfo.summary(model, input_size=(1, 1, 28, 28), device=device, mode="eval")

> 💡 The output layer has **10 neurons** — one *logit* per class. We do **not**
> add a softmax here: the loss function below (`CrossEntropyLoss`) combines
> log-softmax and negative log-likelihood in a numerically stable way. This mirrors Lesson 1, where
> `BCEWithLogitsLoss` handled the sigmoid for us.

## Loss, optimizer, and the training loop ⭐

We reuse the mini-batch approach from Lesson 2. For **each training batch**:

1. **Forward pass** — predictions from the model.
2. **Loss** — how wrong they are (`CrossEntropyLoss` for multi-class).
3. **Backward pass** — `loss.backward()` computes the gradients (autograd).
4. **Update** — `optimizer.step()` nudges the weights.

Call **`optimizer.zero_grad()`** before each batch's backward pass.

After each epoch we measure both losses with the **same fixed weights**, in
**evaluation mode**, using clean training and validation images. This makes the
loss curves easier to compare: dropout is disabled and BatchNorm uses its running
statistics in both passes. These measurements do not update the weights. The
extra training-set evaluation adds one forward-only pass per epoch.

In [ ]:
def evaluate_model(model, loader):
    """Measure mean cross-entropy and accuracy without training the model."""
    model.eval()
    model_device = next(model.parameters()).device
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(model_device), yb.to(model_device)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            count += xb.size(0)
    return total_loss / count, correct / count


def train_model(model, train_loader, val_loader, n_epochs=5, lr=1e-3,
                train_eval_loader=None):
    # For augmented training, supply clean versions of the same training images.
    if train_eval_loader is None:
        train_eval_loader = DataLoader(train_loader.dataset,
                                       batch_size=train_loader.batch_size,
                                       shuffle=False)
    model_device = next(model.parameters()).device
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(model_device), yb.to(model_device)
            optimizer.zero_grad()                 # 0) reset gradients
            logits = model(xb)                    # 1) forward pass
            loss = criterion(logits, yb)          # 2) loss
            loss.backward()                       # 3) backward pass (autograd)
            optimizer.step()                      # 4) update the weights

        # Compare clean training/validation data using the final weights this epoch.
        train_loss, _ = evaluate_model(model, train_eval_loader)
        val_loss, val_acc = evaluate_model(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch+1:2d}/{n_epochs}  "
              f"train_loss={train_loss:.3f}  val_loss={val_loss:.3f}  val_acc={val_acc:.3f}")
    return history

In [ ]:
history = train_model(model, train_loader, val_loader, n_epochs=5, lr=1e-3,
                      train_eval_loader=train_eval_loader)

In [ ]:
def show_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    ax1.plot(epochs, history["train_loss"], label="clean train (eval)")
    ax1.plot(epochs, history["val_loss"], label="validation (eval)")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("Loss")
    ax2.plot(epochs, history["val_acc"], label="val_acc", color="green")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend(); ax2.set_title("Validation accuracy")
    plt.tight_layout(); plt.show()

show_history(history)

> 💡 Both loss curves use clean images and evaluation mode. If validation loss
> rises over several epochs while training loss falls, that is evidence of
> overfitting. A gap alone, or one noisy epoch, is not enough to draw that conclusion.

## 📊 Part 4 — Evaluate on the validation set

Use **validation** accuracy and loss to compare models and choose hyperparameters.
We keep the test set aside until we have chosen a model, then evaluate it once in
"Final test evaluation" below. Looking at test scores while tuning would make the
reported score an unreliable estimate of performance on unseen data.

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(1).cpu().numpy()   # pick the class with the highest logit
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

acc = accuracy_score(all_true, all_preds)
print(f"Validation accuracy: {acc:.3f}")

### Confusion matrix

In [ ]:
cm = confusion_matrix(all_true, all_preds, labels=range(len(class_names)))

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Validation confusion matrix — SimpleCNN")
plt.tight_layout(); plt.show()

# 🔬 Part 5 — Look inside the network

Remember the hand-made filters from Part 1? The network learned its **own** filters
during training. Let's visualise what the convolutional layers produce for a single
image — these outputs are called **feature maps** (or *activations*).

In [ ]:
# Take one validation image and push it through the layers one by one.
sample, sample_label = val_set[0]
sample = sample.unsqueeze(0).to(device)       # add the batch dimension -> (1, 1, 28, 28)

model.eval()
with torch.no_grad():
    act1 = torch.relu(model.conv1(sample))                    # after conv1 -> (1, 32, 26, 26)
    act2 = torch.relu(model.conv2(model.pool(act1)))          # after conv2 -> (1, 16, 11, 11)

print("input image:", class_names[sample_label])
plt.figure(figsize=(3, 3))
plt.imshow(sample.cpu()[0, 0], cmap="gray")
plt.title("Input"); plt.axis("off"); plt.show()

In [ ]:
def show_feature_maps(activation, n_cols=8, title=""):
    activation = activation.detach().cpu()[0]           # drop the batch dim -> (channels, H, W)
    n = activation.shape[0]
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.3, n_rows * 1.3),
                             squeeze=False)
    for i, ax in enumerate(axes.ravel()):
        if i < n:
            ax.imshow(activation[i], cmap="gray")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

show_feature_maps(act1, n_cols=8, title="Feature maps — conv1 (32 filters)")

In [ ]:
show_feature_maps(act2, n_cols=8, title="Feature maps — conv2 (16 filters)")

## 💡 What did we just see?

Each little square is the output of **one learned filter**. Some fire on **edges**,
some on **bright/dark regions**, some on **textures** — exactly the kinds of things
we built by hand in Part 1, except **the network discovered them by itself** from
the data. That is the whole magic of a CNN. ✨

# 🔎 How do we make a CNN better?

Here are three changes to experiment with. Their benefit depends on the task and
training settings, so check the validation results.

### 📒 Batch Normalization
During training, **BatchNorm2d** normalises each channel using statistics over the
batch and spatial dimensions, then applies a **learned scale and bias**. Its final
output therefore does not have to have zero mean and unit variance. In evaluation
mode it uses running statistics collected during training by default.

BatchNorm often makes optimisation faster or more stable and can have a
regularising effect. It does not guarantee less overfitting. In PyTorch, place
**`nn.BatchNorm2d(num_channels)`** after a convolution; see the
[BatchNorm2d documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html).

### 📒 Dropout
**Dropout** randomly zeros a fraction of activations during training and rescales
the remaining ones. This can reduce reliance on particular features and help with
overfitting. In PyTorch, **`nn.Dropout(0.2)`** drops 20% of activations during
`model.train()` and is disabled during `model.eval()`.

### 📒 Mish activation
**Mish** is a smooth alternative to ReLU:
$\text{mish}(x) = x \cdot \tanh(\ln(1+e^{x}))$.
Unlike ReLU, it does not set every negative input to zero. It can still have very
small gradients for strongly negative inputs, so it does not guarantee that
neurons always learn effectively.

In [ ]:
class BetterCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # padding=1 keeps 28x28
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(16)
        self.pool  = nn.MaxPool2d(2)
        self.act   = nn.Mish()                                    # built-in Mish
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(16 * 7 * 7, 64)                  # 28 ->14 ->7  => 16*7*7
        self.dropout = nn.Dropout(0.2)
        self.fc2     = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.pool(self.act(self.bn1(self.conv1(x))))          # 28 -> 14
        x = self.pool(self.act(self.bn2(self.conv2(x))))          # 14 -> 7
        x = self.flatten(x)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

better_model = BetterCNN().to(device)
torchinfo.summary(better_model, input_size=(1, 1, 28, 28), device=device, mode="eval")

In [ ]:
better_history = train_model(better_model, train_loader, val_loader, n_epochs=5, lr=1e-3,
                             train_eval_loader=train_eval_loader)
show_history(better_history)

> 🔎 Compare validation accuracy and the clean train/validation loss curves with
> `SimpleCNN`. Is there evidence of less overfitting?
>
> `BetterCNN` changes padding, activation, BatchNorm, and dropout together. Padding
> also increases the flattened size and total parameters (**31,258 → 55,930**).
> An improvement cannot be attributed to regularisation alone. To investigate
> causes, change one component at a time, keep training settings comparable, and
> repeat experiments with several random seeds.

In [ ]:
better_val_loss, better_val_acc = evaluate_model(better_model, val_loader)
print(f"BetterCNN validation loss: {better_val_loss:.3f}, accuracy: {better_val_acc:.3f}")

## 📒 Data augmentation

Deep learning is hungry for data, and collecting more is expensive. **Data
augmentation** is a cheap trick: create modified copies of existing images (flips,
small rotations, shifts) so the model sees more variety and may generalise better.

In PyTorch, augmentation is just a **transform** applied while loading. It runs
**on the fly** — a different random version every epoch. Note we only augment the
**training** data, never validation/test. The clean training loader used for
loss comparisons also stays unchanged.

Augmentations should preserve the label. Horizontal flips and small rotations
are reasonable experiments for clothing, but they need not help every model.
In particular, five epochs with augmentation can score lower than the baseline;
compare validation results and learning curves instead of assuming improvement.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),          # clothes are roughly left-right symmetric
    transforms.RandomRotation(10),              # small +/- 10 degree rotation
    transforms.ToTensor(),
])

# Rebuild ONLY the training dataset with augmentation (val/test stay clean)
aug_full_train = datasets.FashionMNIST(root="data", train=True, download=True,
                                       transform=train_transform)
# Reuse exactly the original training indices; validation remains separate.
aug_train_set = Subset(aug_full_train, train_set.indices)
aug_train_loader = DataLoader(aug_train_set, batch_size=BATCH_SIZE, shuffle=True)

# Show a few augmented samples
aug_images, aug_labels = next(iter(aug_train_loader))
plt.figure(figsize=(9, 2))
for i in range(min(6, len(aug_images))):
    plt.subplot(1, 6, i + 1)
    plt.imshow(aug_images[i][0], cmap="gray"); plt.axis("off")
    plt.xlabel(class_names[aug_labels[i]])
plt.suptitle("Augmented training images")
plt.tight_layout(); plt.show()

In [ ]:
# Train a fresh model on the augmented data (reusing the same loop again).
aug_model = SimpleCNN().to(device)
aug_history = train_model(aug_model, aug_train_loader, val_loader, n_epochs=5, lr=1e-3,
                          train_eval_loader=train_eval_loader)
show_history(aug_history)

In [ ]:
aug_val_loss, aug_val_acc = evaluate_model(aug_model, val_loader)
print(f"Augmented SimpleCNN validation loss: {aug_val_loss:.3f}, accuracy: {aug_val_acc:.3f}")

## 🚀 What if the dataset is too big to fit in memory?

So far the whole dataset lived in RAM. For large datasets you instead keep images
on **disk** in a folder-per-class structure and load them **lazily** in batches.

```
data_dir/
├── 0/  (all images of class 0)
├── 1/
└── ...
```

In PyTorch we use
**`torchvision.datasets.ImageFolder`** + a `DataLoader`. It infers the label from
the folder name and reads images from disk when samples are requested. The
default loader converts images to RGB, so we explicitly convert them back to
grayscale for our one-channel CNNs. Numeric folder names are sorted as strings;
the labels match FashionMNIST here because the folders are exactly `0`–`9`.

The next cell exports a small sample only to demonstrate the folder format.
It still loads the original FashionMNIST data into RAM; for a genuinely large
dataset, start with images already stored on disk. This exported sample is not
used in the training/validation experiment above.

In [ ]:
import os
from PIL import Image

# Save a small slice of FashionMNIST to disk as PNGs, one folder per class.
OUTPUT_DIR = "fashion_dir/train"
raw = datasets.FashionMNIST(root="data", train=True, download=True)  # PIL images, no transform

n_to_save = 2000                                   # keep it small & fast for the demo
for i in range(n_to_save):
    image, label = raw[i]                          # image is a PIL Image
    label_dir = os.path.join(OUTPUT_DIR, str(label))
    os.makedirs(label_dir, exist_ok=True)
    image.save(os.path.join(label_dir, f"{i:05d}.png"))

print("saved", n_to_save, "images to", OUTPUT_DIR)

In [ ]:
# ImageFolder loads RGB by default; convert back to one channel for our CNNs.
folder_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])
folder_dataset = datasets.ImageFolder(root=OUTPUT_DIR, transform=folder_transform)
folder_loader = DataLoader(folder_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("classes found:", folder_dataset.classes)
xb, yb = next(iter(folder_loader))
print("batch shape:", xb.shape)                    # (batch, 1, 28, 28)

# Activities ✏️

Now it is your turn to change the code. Copy cells from above as a starting point.

## Activity 1 — Train longer & tune 🎚️

Train a **fresh** `SimpleCNN` for each experiment and try to improve
**validation accuracy**:

- increase `n_epochs` (e.g. `10`–`20`),
- try a different learning rate (e.g. `5e-4`, `3e-3`),
- try a different `BATCH_SIZE`. Smaller batches use less memory and produce
  noisier gradient estimates; they are not automatically more stable or faster.

After changing `BATCH_SIZE`, **rebuild the DataLoaders**: assigning a new value
alone does not change existing loaders. Keep the train/validation split fixed.

Which combination gives the best validation accuracy? Finish tuning before
running "Final test evaluation". Select the configuration using validation results
and use the test set only to report the final score.

## Activity 2 — Change the architecture 🏗️

Modify the network and see what happens. Try **at least two** of these:

- change the number of filters (e.g. `Conv2d(1, 64, 3)`), and update the next
  convolution to accept those 64 channels,
- add a **third** convolutional block,
- change the kernel size to `5` (remember to recompute the `Linear` input size —
  or use `padding=2` for a 5×5 kernel with stride 1 and dilation 1).

Check every intermediate shape and update `fc1.in_features` when needed.

## Activity 3 — Regularise it 🛡️

Start from `BetterCNN` (BatchNorm + Dropout + Mish). Train it for more epochs and
compare validation performance and the clean train/validation loss curves with
`SimpleCNN`. Remember that the two models differ in several ways.

To isolate dropout's effect, keep the architecture, data split, optimiser, and
epoch count fixed, and vary only the dropout rate (`0.0`, `0.1`, `0.3`, `0.5`).
Train a fresh model for each run and repeat with several seeds. Does a particular
rate improve validation performance or reduce evidence of overfitting?

## Activity 4 — Try harder datasets 🧪

**CIFAR-10 ≥ 80%** — colour `32×32` images in 10 classes. **(1p)**

- Load with `datasets.CIFAR10(...)` and make a validation split from its training
  set. Use the CIFAR-10 class names when plotting labels or confusion matrices.
- Input now has **3 channels**: change the first convolution to
  `nn.Conv2d(3, 32, kernel_size=3)` (retain `padding=1` if using `BetterCNN`).
- Recompute the flattened size. With all other layers unchanged, `SimpleCNN`
  needs `nn.Linear(16 * 6 * 6, 64)`; `BetterCNN` needs
  `nn.Linear(16 * 8 * 8, 64)`.
- Train for more epochs (e.g. 20–50) and experiment with capacity, BatchNorm,
  dropout, and label-preserving augmentation. These changes do not guarantee 80%.
- Tune on **validation** results, then evaluate the selected model once on the
  held-out test set to check the target.

## 📊 Final test evaluation

Run this section **after finishing model selection**, including any activities
that change the FashionMNIST models. The example below chooses among the three
trained demonstration models by their final validation accuracy. To include an
activity model, add it to `candidates` before selection. All candidates must use
the same FashionMNIST training/validation split; keep CIFAR-10 experiments separate.

The test labels are used only after selection. Report the result without using it
to choose further changes; otherwise the test set becomes part of the tuning loop.

In [ ]:
candidates = {
    "SimpleCNN": model,
    "BetterCNN": better_model,
    "SimpleCNN + augmentation": aug_model,
}
validation_scores = {}
for name, candidate in candidates.items():
    _, val_acc = evaluate_model(candidate, val_loader)
    validation_scores[name] = val_acc
    print(f"{name}: validation accuracy = {val_acc:.3f}")

selected_name = max(validation_scores, key=validation_scores.get)
selected_model = candidates[selected_name]
print("Selected using validation accuracy:", selected_name)

In [ ]:
selected_model.eval()
selected_device = next(selected_model.parameters()).device
all_test_preds, all_test_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = selected_model(xb.to(selected_device))
        all_test_preds.extend(logits.argmax(1).cpu().numpy())
        all_test_true.extend(yb.cpu().numpy())

print(f"{selected_name} — final test accuracy: "
      f"{accuracy_score(all_test_true, all_test_preds):.3f}")
cm = confusion_matrix(all_test_true, all_test_preds, labels=range(len(class_names)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Test confusion matrix — {selected_name}")
plt.tight_layout(); plt.show()